In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a batched matrix multiplication in FP32. Given a batch of matrices <code>A</code> of shape <code>[B, M, K]</code> and a batch of matrices <code>B</code> of shape <code>[B, K, N]</code>, compute the output batch <code>C</code> of shape <code>[B, M, N]</code> such that for each batch index <code>b</code>:
  $$
    C_b = A_b \times B_b
  $$
  All matrices are stored in row-major order and use 32-bit floating point numbers (FP32).
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>C</code> array</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
B = 2, M = 2, K = 3, N = 2
A = [
  [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]],
  [[7.0, 8.0, 9.0], [10.0, 11.0, 12.0]]
]
B = [
  [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]],
  [[6.0, 5.0], [4.0, 3.0], [2.0, 1.0]]
]
Output:
C = [
  [[22.0, 28.0], [49.0, 64.0]],
  [[92.0, 68.0], [128.0, 95.0]]
]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>B</code> &le; 128</li>
  <li>1 &le; <code>M</code>, <code>N</code>, <code>K</code> &le; 1024</li>

  <li>Performance is measured with <code>K</code> = 256, <code>M</code> = 256, <code>N</code> = 256</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// A, B, C are device pointers
extern "C" void solve(const float* A, const float* B, float* C, int BATCH, int M, int N, int K) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# A, B, C are tensors on the GPU
@cute.jit
def solve(
    A: cute.Tensor,
    B: cute.Tensor,
    C: cute.Tensor,
    BATCH: cute.Int32,
    M: cute.Int32,
    N: cute.Int32,
    K: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# A, B are tensors on the GPU
@jax.jit
def solve(A: jax.Array, B: jax.Array, BATCH: int, M: int, N: int, K: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    A: UnsafePointer[Float32, MutExternalOrigin],
    B: UnsafePointer[Float32, MutExternalOrigin],
    C: UnsafePointer[Float32, MutExternalOrigin],
    BATCH: Int32,
    M: Int32,
    N: Int32,
    K: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# A, B, C are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, C: torch.Tensor, BATCH: int, M: int, N: int, K: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# a, b, c are tensors on the GPU
def solve(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor, BATCH: int, M: int, N: int, K: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Batched Matrix Multiplication",
            atol=1e-5,
            rtol=1e-5,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self, A: torch.Tensor, B: torch.Tensor, C: torch.Tensor, BATCH: int, M: int, N: int, K: int
    ):
        # A: (BATCH, M, K), B: (BATCH, K, N), C: (BATCH, M, N)
        A = A.view(BATCH, M, K)
        B = B.view(BATCH, K, N)
        C.copy_(torch.bmm(A, B))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "A": (ctypes.POINTER(ctypes.c_float), "in"),
            "B": (ctypes.POINTER(ctypes.c_float), "in"),
            "C": (ctypes.POINTER(ctypes.c_float), "out"),
            "BATCH": (ctypes.c_int, "in"),
            "M": (ctypes.c_int, "in"),
            "N": (ctypes.c_int, "in"),
            "K": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        BATCH, M, K, N = 2, 2, 3, 2
        A = torch.tensor(
            [[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], [[7.0, 8.0, 9.0], [10.0, 11.0, 12.0]]],
            device="cuda",
            dtype=dtype,
        )
        B = torch.tensor(
            [[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], [[6.0, 5.0], [4.0, 3.0], [2.0, 1.0]]],
            device="cuda",
            dtype=dtype,
        )
        C = torch.empty(BATCH, M, N, device="cuda", dtype=dtype)
        return {"A": A, "B": B, "C": C, "BATCH": BATCH, "M": M, "N": N, "K": K}

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        device = "cuda"
        tests = []

        # 1. basic_example
        A1 = torch.tensor(
            [[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], [[7.0, 8.0, 9.0], [10.0, 11.0, 12.0]]],
            device=device,
            dtype=dtype,
        ).flatten()
        B1 = torch.tensor(
            [[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], [[6.0, 5.0], [4.0, 3.0], [2.0, 1.0]]],
            device=device,
            dtype=dtype,
        ).flatten()
        C1 = torch.empty((2, 2, 2), device=device, dtype=dtype)
        tests.append({"A": A1, "B": B1, "C": C1, "BATCH": 2, "M": 2, "N": 2, "K": 3})

        # 2. single_batch
        A2 = torch.tensor(
            [[[1.0, 0.0, 2.0], [0.0, 1.0, 2.0], [2.0, 1.0, 0.0]]], device=device, dtype=dtype
        ).flatten()
        B2 = torch.tensor(
            [[[2.0, 1.0, 0.0], [1.0, 2.0, 0.0], [0.0, 1.0, 2.0]]], device=device, dtype=dtype
        ).flatten()
        C2 = torch.empty((1, 3, 3), device=device, dtype=dtype)
        tests.append({"A": A2, "B": B2, "C": C2, "BATCH": 1, "M": 3, "N": 3, "K": 3})

        # 3. batch_4_small
        A3 = torch.empty((4, 2, 2), device=device, dtype=dtype).uniform_(-1.0, 1.0)
        B3 = torch.empty((4, 2, 2), device=device, dtype=dtype).uniform_(-1.0, 1.0)
        C3 = torch.empty((4, 2, 2), device=device, dtype=dtype)
        tests.append({"A": A3, "B": B3, "C": C3, "BATCH": 4, "M": 2, "N": 2, "K": 2})

        # 4. batch_8_rectangular
        A4 = torch.empty((8, 4, 2), device=device, dtype=dtype).uniform_(-10.0, 10.0)
        B4 = torch.empty((8, 2, 3), device=device, dtype=dtype).uniform_(-10.0, 10.0)
        C4 = torch.empty((8, 4, 3), device=device, dtype=dtype)
        tests.append({"A": A4, "B": B4, "C": C4, "BATCH": 8, "M": 4, "N": 3, "K": 2})

        # 5. batch_16_large
        A5 = torch.empty((16, 16, 16), device=device, dtype=dtype).uniform_(-1.0, 1.0)
        B5 = torch.empty((16, 16, 16), device=device, dtype=dtype).uniform_(-1.0, 1.0)
        C5 = torch.empty((16, 16, 16), device=device, dtype=dtype)
        tests.append({"A": A5, "B": B5, "C": C5, "BATCH": 16, "M": 16, "N": 16, "K": 16})

        # 6. batch_2_non_square
        A6 = torch.empty((2, 8, 4), device=device, dtype=dtype).uniform_(-5.0, 5.0)
        B6 = torch.empty((2, 4, 6), device=device, dtype=dtype).uniform_(-5.0, 5.0)
        C6 = torch.empty((2, 8, 6), device=device, dtype=dtype)
        tests.append({"A": A6, "B": B6, "C": C6, "BATCH": 2, "M": 8, "N": 6, "K": 4})

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        BATCH, M, N, K = 32, 256, 256, 256  # Match speed_test.json
        A = torch.empty(BATCH, M, K, device="cuda", dtype=dtype).uniform_(
            -10.0, 10.0
        )  # Match range
        B = torch.empty(BATCH, K, N, device="cuda", dtype=dtype).uniform_(
            -10.0, 10.0
        )  # Match range
        C = torch.empty(BATCH, M, N, device="cuda", dtype=dtype)
        return {"A": A, "B": B, "C": C, "BATCH": BATCH, "M": M, "N": N, "K": K}


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
